# 03b - Neighbourhood Radius Sweep

Runs the **baseline model only** across a set of feature-extraction radii for every dataset, and reports test accuracy per radius

In [ ]:
import sys
from pathlib import Path

current_notebook_dir = Path.cwd()
if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import gc
import copy
import time

import numpy as np
import pandas as pd

from src.utils import set_global_seed, load_config, get_logger, configure_gpu_memory_growth
configure_gpu_memory_growth()

from data.paths import get_dataset_config
from src.io_utils import load_raw_point_file
from src.features import extract_features, DEFAULT_FEATURE_NAMES
from src.classes import get_class_registry
from src.normalization import fit_scaler, apply_scaler
from src.losses import get_sparse_categorical_crossentropy_loss
from src.model import build_model, compile_model, train_model, make_tf_dataset

logger = get_logger("09_radius_sweep")

## Sweep configuration

`RADII` is read per dataset from `configs/datasets.yaml -> feature_radii`

In [ ]:
DATASETS = ["vaihingen", "ahn3", "paris_lille", "toronto3d"]

RADII_OVERRIDE = None
SWEEP_EPOCHS = 5 #Small training
OUTPUT_CSV = project_root / "results" / "radius_sweep.csv"

SCRATCH_CKPT = project_root / "checkpoints" / "_radius_sweep"
SCRATCH_CKPT.mkdir(parents=True, exist_ok=True)

base_config = load_config("default")

sweep_config = copy.deepcopy(base_config)
sweep_config["training"]["num_epochs"] = SWEEP_EPOCHS
sweep_config["training"]["early_stopping"]["enabled"] = False

print(f"Datasets     : {DATASETS}")
print(f"Epochs/run   : {SWEEP_EPOCHS}")
print(f"Batch size   : {sweep_config['training']['batch_size']}")
print(f"Seed         : {sweep_config['experiment']['seed']}")
print(f"Output       : {OUTPUT_CSV}")

In [ ]:
for ds in DATASETS:
    cfg = get_dataset_config(ds)
    split_cfg = cfg["split"]
    radii = RADII_OVERRIDE or cfg.get("feature_radii", [cfg.get("feature_radius", 0.25)])
    print(f"{ds}")
    print(f"   files  : {cfg['files']}")
    print(f"   split  : {split_cfg}")
    print(f"   radii  : {radii}")
    print()

## Helper : build a train/test split in memory at a given radius
- Extract -> file-based split -> drop NaN flagged points -> map class indices -> fit scaler on train only
- Everything in memory

In [ ]:
def _split_file_lists(dataset: str):
    split_cfg = get_dataset_config(dataset)["split"]
    if split_cfg.get("strategy", "file_based") != "file_based":
        raise ValueError(
            f"[{dataset}] split strategy is {split_cfg.get('strategy')!r}; "
            "the radius sweep requires the canonical file_based split."
        )

    
    train_files = split_cfg.get("train_files") or split_cfg.get("train")
    test_files = split_cfg.get("test_files") or split_cfg.get("test")
    if not train_files or not test_files:
        raise KeyError(
            f"[{dataset}] could not find train/test file lists in split config: {split_cfg}"
        )
    overlap = set(train_files) & set(test_files)
    assert not overlap, f"[{dataset}] train/test file overlap: {overlap}"
    return list(train_files), list(test_files)

In [ ]:
def build_split_at_radius(dataset: str, radius: float, config: dict):
    train_files, test_files = _split_file_lists(dataset)
    feature_columns = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES

    frames = {"train": [], "test": []}
    nan_counts = {"train": [0, 0], "test": [0, 0]}   # [n_flagged, n_total]

    for split_name, file_list in (("train", train_files), ("test", test_files)):
        for file_name in file_list:
            raw_df = load_raw_point_file(dataset, file_name)
            feats = extract_features(raw_df, dataset=dataset, radius=radius)
            del raw_df

            feats["__source_file__"] = file_name

            nan_counts[split_name][0] += int(feats["__nan_feature__"].sum())
            nan_counts[split_name][1] += len(feats)

            feats = feats.loc[~feats["__nan_feature__"]].reset_index(drop=True)
            frames[split_name].append(feats)
            gc.collect()

    train_df = pd.concat(frames["train"], ignore_index=True)
    test_df = pd.concat(frames["test"], ignore_index=True)
    del frames
    gc.collect()

    class_registry = get_class_registry(dataset)
    train_df["class_idx"] = class_registry.map_labels(train_df["label"].values)
    test_df["class_idx"] = class_registry.map_labels(test_df["label"].values)

    # scaler fitted on train only
    scaler = fit_scaler(train_df, feature_columns, config)
    train_scaled = apply_scaler(train_df, scaler, feature_columns)
    test_scaled = apply_scaler(test_df, scaler, feature_columns)
    del train_df, test_df
    gc.collect()

    X_train = train_scaled[feature_columns].values.astype(np.float32)
    y_train = train_scaled["class_idx"].values.astype(np.int32)
    X_test = test_scaled[feature_columns].values.astype(np.float32)
    y_test = test_scaled["class_idx"].values.astype(np.int32)
    del train_scaled, test_scaled
    gc.collect()

    diagnostics = {
        "pct_nan_train": round(100 * nan_counts["train"][0] / max(nan_counts["train"][1], 1), 2),
        "pct_nan_test": round(100 * nan_counts["test"][0] / max(nan_counts["test"][1], 1), 2),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "num_classes": class_registry.num_classes,
    }
    return X_train, y_train, X_test, y_test, diagnostics

## Helper : train the baseline and get test acc

In [ ]:
def train_and_score(X_train, y_train, X_test, y_test, num_classes, config):
    import tensorflow as tf
    tf.keras.backend.clear_session()
    set_global_seed(config["experiment"]["seed"])

    model = build_model(
        input_dim=X_train.shape[1],
        num_classes=num_classes,
        config=config,
    )
    model = compile_model(model, config, loss=get_sparse_categorical_crossentropy_loss())

    train_ds = make_tf_dataset(
        X_train, y_train,
        batch_size=config["training"].get("batch_size", 512),
        shuffle=True,
        seed=config["experiment"]["seed"],
    )

    model, history = train_model(
        model, train_ds, None, config,
        checkpoint_path=SCRATCH_CKPT / "sweep.keras",
    )

    y_pred = np.argmax(
        model.predict(X_test, batch_size=config["training"].get("batch_size", 512), verbose=0),
        axis=1,
    )
    test_accuracy = float((y_pred == y_test).mean())
    train_accuracy = float(history.history["accuracy"][-1])

    del model, train_ds, y_pred
    tf.keras.backend.clear_session()
    gc.collect()

    return test_accuracy, train_accuracy

## Full sweep

In [ ]:
def load_existing_results(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Resuming: {len(df)} run(s) already in {path.name}")
        return df
    return pd.DataFrame()


OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df = load_existing_results(OUTPUT_CSV)
done = set()
if len(results_df):
    done = {(r.dataset, round(float(r.radius), 4)) for r in results_df.itertuples()}

for dataset in DATASETS:
    cfg = get_dataset_config(dataset)
    radii = RADII_OVERRIDE or cfg.get("feature_radii", [cfg.get("feature_radius", 0.25)])

    for radius in radii:
        key = (dataset, round(float(radius), 4))
        if key in done:
            print(f"[skip] {dataset} r={radius} already recorded")
            continue

        print("=" * 70)
        print(f"[run] dataset={dataset}  radius={radius}  epochs={SWEEP_EPOCHS}")
        print("=" * 70)
        t0 = time.time()

        try:
            X_train, y_train, X_test, y_test, diag = build_split_at_radius(
                dataset, radius, sweep_config
            )
            print(f"   train={diag['n_train']:,}  test={diag['n_test']:,}  "
                  f"NaN-flagged train={diag['pct_nan_train']}%  test={diag['pct_nan_test']}%")

            test_acc, train_acc = train_and_score(
                X_train, y_train, X_test, y_test, diag["num_classes"], sweep_config
            )

            row = {
                "dataset": dataset,
                "radius": float(radius),
                "test_accuracy": round(test_acc, 4),
                "train_accuracy": round(train_acc, 4),
                "epochs": SWEEP_EPOCHS,
                "n_train": diag["n_train"],
                "n_test": diag["n_test"],
                "pct_nan_train": diag["pct_nan_train"],
                "pct_nan_test": diag["pct_nan_test"],
                "runtime_sec": round(time.time() - t0, 1),
            }
            print(f"   -> test accuracy {test_acc:.4f}  ({row['runtime_sec']}s)")

        except Exception as exc:
            logger.error(f"[{dataset} r={radius}] FAILED: {exc}")
            row = {
                "dataset": dataset,
                "radius": float(radius),
                "test_accuracy": np.nan,
                "train_accuracy": np.nan,
                "epochs": SWEEP_EPOCHS,
                "n_train": np.nan,
                "n_test": np.nan,
                "pct_nan_train": np.nan,
                "pct_nan_test": np.nan,
                "runtime_sec": round(time.time() - t0, 1),
            }

        finally:
            for name in ("X_train", "y_train", "X_test", "y_test"):
                if name in dir():
                    del globals()[name]
            gc.collect()

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
        results_df.to_csv(OUTPUT_CSV, index=False)

print()
print(f"Sweep complete. {len(results_df)} run(s) -> {OUTPUT_CSV}")

## Results

In [ ]:
results_df = pd.read_csv(OUTPUT_CSV).sort_values(["dataset", "radius"])
results_df

In [ ]:
pivot = results_df.pivot(index="radius", columns="dataset", values="test_accuracy")
pivot = pivot[[d for d in DATASETS if d in pivot.columns]]
print("Test accuracy by neighbourhood radius\n")
print(pivot.to_string())
print("\nBest radius per dataset:")
print(pivot.idxmax().to_string())

## Clean up scratch checkpoints
- The sweep writes a single throwaway weights file per run. Remove it here so nothing from this notebook persists apart from the results CSV

In [ ]:
import shutil

if SCRATCH_CKPT.exists():
    shutil.rmtree(SCRATCH_CKPT)
    print(f"Removed {SCRATCH_CKPT}")
else:
    print("Nothing to clean up.")